# 072 — Proyecto: asistente multimodal accesible

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**WCAG (W3C):** cuatro principios **POUR** — Perceptible (alt-text 1.1.1, subtítulos 1.2.2,
contraste 1.4.3), Operable (teclado 2.1.1), Comprensible, Robusto (ARIA) — con niveles A /
AA (objetivo habitual) / AAA. Contraste:
`ratio = (L_claro + 0.05)/(L_oscuro + 0.05)`, AA ≥ 4.5, AAA ≥ 7.

**Arquitectura del asistente:** entrada audio→ASR (067) e imagen→VLM (069), núcleo de
fusión y diálogo (070), salida TTS (068) + pantalla + subtítulos. El **presupuesto de
latencia** (~1.5 s conversacional) se reparte por etapa y la más lenta domina. Cada pieza
falla distinto: WER por subgrupo (ASR), **alucinaciones** (VLM), normalización (TTS).

**Evaluación:** métricas por subgrupo + auditoría WCAG + **pruebas con usuarios de
tecnología asistiva** ("nada sobre nosotros sin nosotros"). El riesgo específico: una
descripción alucinada entregada con voz segura a quien **no puede verificarla** — mitigar
con incertidumbre explícita, rechazo ante baja confianza y límites de uso declarados.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** (a) `(1.0+0.05)/(0.25+0.05) = 1.05/0.30 = 3.5` → **falla AA**: ese gris
"elegante" es ilegible para baja visión. (b) `(1.05)/(0.15) = 7.0` → pasa AA y justo
alcanza AAA (≥ 7).

**Ejercicio 2.** Total = `350+800+150+300+100 = 1700 ms` → 200 ms sobre presupuesto. La VLM
domina (47 % del total): recortarle 200 ms (modelo menor, imagen reducida, caché) resuelve
todo el exceso; recortar 40 ms a cada etapa exigiría optimizar cinco sistemas distintos con
mucho menos margen cada uno. En un pipeline en serie, la etapa dominante es donde el
esfuerzo rinde más.

**Ejercicio 3.** WER_A = 30/600 = **5 %**; WER_B = 45/300 = **15 %**; global = 75/900 ≈
**8.3 %**. El global parece aceptable, pero el grupo B (¿un acento, un grupo etario?)
recibe un servicio 3× peor — y como B aporta menos palabras al total, el promedio lo
invisibiliza. Decisión de producto: recolectar datos del subgrupo B y fijar un umbral de
calidad **por subgrupo**, no global, antes de lanzar.

**Ejercicio 4.** Ejemplo de respuesta: **Perceptible** — toda salida hablada tiene par
visual (subtítulos + texto en pantalla con contraste ≥ 4.5:1). **Operable** — el flujo
completo funciona por teclado/gestos, sin depender de apuntar la cámara con precisión y sin
límites de tiempo rígidos. **Comprensible** — respuestas en lenguaje claro, con la
incertidumbre declarada ("no puedo leer la fecha; ¿otra foto?"). **Robusto** — interfaz
compatible con lectores de pantalla (roles y etiquetas ARIA correctos) probada con al menos
dos lectores reales.


In [ ]:
result = run_lab("capstone", seed=72)
assert result["kind"] == "capstone"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicios 1-3 — verificados con código
def contraste(l_claro, l_oscuro):
    return (l_claro + 0.05) / (l_oscuro + 0.05)

def wer(errores, n):
    return errores / n

print("contraste L=0.25:", round(contraste(1.0, 0.25), 2), "→ AA:", contraste(1.0, 0.25) >= 4.5)
print("contraste L=0.10:", round(contraste(1.0, 0.10), 2), "→ AAA:", round(contraste(1.0, 0.10), 2) >= 7)

etapas = {"ASR": 350, "VLM": 800, "composición": 150, "TTS": 300, "red": 100}
total = sum(etapas.values())
print("latencia total:", total, "ms | exceso:", total - 1500, "ms | dominante:",
      max(etapas, key=etapas.get))

print("WER A:", wer(30, 600), "| WER B:", wer(45, 300), "| global:", round(wer(75, 900), 3))


## Reflexión

1. Tu asistente lee una etiqueta de medicamento con confianza baja. ¿Qué respuesta diseñas
   para un usuario ciego, y por qué "adivinar con voz segura" es peor que negarse y pedir
   otra foto?
2. La auditoría WCAG AA pasó al 100 %, pero los usuarios de lector de pantalla abandonan la
   app en la primera semana. ¿Qué falló en el proceso de evaluación y cómo lo corriges?
3. Identifica un ejemplo del efecto bordillo en tu propio uso diario de tecnología. ¿Qué te
   dice sobre el retorno de diseñar accesible desde el inicio?
